# Cleaning data for training BERT models

We want to clean the API returned tsv file containing 2023-24 attributions and their tasks. We want to use this file for BERT model training, but, some task names in the original file do not exist in our survey (e.g., "other", or "notebook-keeping").

For the BERT training, we only want to keep the tasks that are present in our survey (TASK_CLASSES containing 15 classes as they were named in the survey). So, we will filter out tasks that are not in TASK_CLASSES from the original tsv file. Also, since the 2023-24 attributions have a lot of null Task descriptions, we will drop such rows from the cleaned dataset. Finally, we will save the cleaned dataframe to a new tsv file that will be used for BERT training (`train_data_cleaned.tsv`).

In [2]:
import pandas as pd
import re

In [2]:
# Config

INPUT_TSV_2023_24 = "../data/attributions/2023_2024_attributions/attributions_2023_2024.tsv"
INPUT_TSV_2025 = "../data/attributions/2025_attributions/attributions_anonymized_2025.tsv"
OUTPUT_TSV = "../data/attributions/2022_attributions/bert/bert_training_data/cleaned_2023_24_25.tsv" # one row per task name - Task description in one row is related to just one task name
OUTPUT_MERGED_TASKS_TSV = "../data/attributions/2022_attributions/bert/bert_training_data/merged_tasks_2023_24_25.tsv" # one row per list of task names - Task description in one row is related to multiple task names

TASK_CLASSES = {
    "software",
    "conceptualization",
    "public engagement",
    "writing",
    "investigation",
    "lab maintenance",
    "hardware",
    "project administration",
    "entrepreneurship",
    "fundraising",
    "background research",
    "safety",
    "analysis",
    "visualization",
    "data curation",
}

In [3]:
# Check which task classes are present in the original data

original_attributions_2023_24 = pd.read_csv(INPUT_TSV_2023_24, sep="\t")
original_attributions_2025 = pd.read_csv(INPUT_TSV_2025, sep="\t")

original_attributions_2023_24["Task"].value_counts()

Task
writing                   9612
conceptualization         9046
background-research       8820
investigation             8496
public-engagement         7200
analysis                  6729
visualization             6032
project-administration    5219
notebook-keeping          4569
safety                    3728
data-curation             3625
other                     3069
fundraising               2851
wiki-coding               2259
software                  1677
entrepreneurship          1527
hardware                   933
Name: count, dtype: int64

We can notice that "lab maintenance" does not exist in the 2023-24 data, and 2025 as well. So, "lab maitenance" cannot be used in training the BERT from these years, and also, since survey data in 2023-25 is connected with attributions, analysis on this task for these years cannot be done. In 2023-25 combined survey and attributions data had "lab maitenance" as TaskPerformed == 0. We will change that to null here, so we don't go back to the previous notebooks.

Besides, there are extra tasks: "other", "notebook-keeping", "wiki coding". Like we did in 2025, we will rename "notebook-keeping" to "writing", and "wiki-coding" to "software", and drop "other". We will also merge all 2023, 2024 with all 2025 attributions to have more training data.

In [8]:
survey_atributions_2023 = pd.read_csv("../results/survey_tasks_and_attributions/2023_survey_tasks_and_attributions_combined.tsv", sep="\t")
survey_atributions_2023_ordinal = pd.read_csv("../results/survey_tasks_and_attributions/2023_survey_tasks_and_attributions_ordinal.tsv", sep="\t")
survey_atributions_2025 = pd.read_csv("../results/survey_tasks_and_attributions/2025_survey_tasks_and_attributions_combined.tsv", sep="\t")

# For Task = "Lab Maintenance", set TaskPerformed to null (instead of zero)
dfs = [survey_atributions_2023, survey_atributions_2023_ordinal, survey_atributions_2025]

for df in dfs:
    df.loc[df["Task"] == "Lab Maintenance", "TaskPerformed"] = None

# Save them
survey_atributions_2023.to_csv("../results/survey_tasks_and_attributions/2023_survey_tasks_and_attributions_combined.tsv", sep="\t", index=False)
survey_atributions_2023_ordinal.to_csv("../results/survey_tasks_and_attributions/2023_survey_tasks_and_attributions_ordinal.tsv", sep="\t", index=False)
survey_atributions_2025.to_csv("../results/survey_tasks_and_attributions/2025_survey_tasks_and_attributions_combined.tsv", sep="\t", index=False)

In [4]:
# Merge 2023-24 and 2025 data
original_attributions = pd.concat([original_attributions_2023_24, original_attributions_2025], ignore_index=True)
len(original_attributions)

137036

In [17]:
# Clean Task column - we did the same in 13_clean_2025_survey_data.ipynb
 
'''
- Convert Task to lowercase
- Replace dashes with spaces
- Strip leading and trailing spaces
- Change some task names to fit the survey
'''

def clean_tasks_in_df(df: pd.DataFrame) -> pd.DataFrame:
    if "Task" in df.columns:
        df = df.copy()
        df["Task"] = (
            df["Task"]
            .astype(str)
            .str.lower()
            .str.replace("-", " ", regex=False)
            .str.strip()
        )

        # Specific name normalizations
        df["Task"] = (
            df["Task"]
            .replace({
                "notebook keeping": "writing",
                "wiki coding": "software"
            })
        )

        # Drop rows with tasks not in TASK_CLASSES (now only "other" but we will make the function reusable)
        df = df[df["Task"].isin(TASK_CLASSES)]

    return df

In [29]:
# Clean TaskDescription column - remove task labels and numbering 

tasks_to_look_for = {
    "software", "conceptualization", "public engagement", "writing",
    "investigation", "lab maintenance", "hardware", "project administration",
    "entrepreneurship", "fundraising", "background research", "safety",
    "analysis", "visualization", "data curation", "wiki coding", "notebook keeping"
} # same as TASK_CLASSES but including old task names

def clean_task_description_in_df(df):
    # Sort labels by length descending to find longest match first

    sorted_labels = sorted([re.escape(label) for label in tasks_to_look_for], key=len, reverse=True)
    labels_pattern = '|'.join(sorted_labels)
    
    full_pattern = rf'([\(\d\)\.\s（）]*)(?:{labels_pattern})\s*[:\-：—－—]{{1,}}\s*'

    def clean_text(text):
        if not isinstance(text, str):
            return text
        
        cleaned = re.sub(full_pattern, ' ', text, flags=re.IGNORECASE)
        
        return re.sub(r'\s+', ' ', cleaned).strip()

    df['TaskDescription'] = df['TaskDescription'].apply(clean_text)
    return df

# Example usage:
example_data = {
    'TaskDescription': [
        "(1)Writing: Helped find information for the description page.(2)Safety- Ensured that all lab members followed safety protocols.",
        "Software- Developed metabolic modelling of TPA degradation on MATLAB",
        "(1) Conceptualization - As a supervisor of our hardware team...",
        "(1) Hardware-he wrote codes and established the portable device.",
        "1 Wiki Coding - Wikipedia page layout, graphic design",
        "1. project administration: Planned the project timeline.",
        "Wiki Coding--She participates in the construction of the wiki.",
        "Investigation：Performing the experiments and/or collecting data/evidence",
        "Conceptualization——Provided initial guidance during project brainstorming",
        "（1）Visualization—She is responsible for the design and production of the team uniforms",

        
    ]
}

example_df = pd.DataFrame(example_data)
example_df = clean_task_description_in_df(example_df)
print(example_df['TaskDescription'].tolist())

['Helped find information for the description page Ensured that all lab members followed safety protocols.', 'Developed metabolic modelling of TPA degradation on MATLAB', 'As a supervisor of our hardware team...', 'he wrote codes and established the portable device.', 'Wikipedia page layout, graphic design', 'Planned the project timeline.', 'She participates in the construction of the wiki.', 'Performing the experiments and/or collecting data/evidence', 'Provided initial guidance during project brainstorming', 'She is responsible for the design and production of the team uniforms']


The previous function is used to clean the task descriptions because a lot of descriptions include the names of the tasks, with numbers, colons and/or dashes. We do not want that in the task description so the model does not learn based on it.

In [30]:
cleaned_tasks_df = clean_tasks_in_df(original_attributions)
cleaned_tasks_df = clean_task_description_in_df(cleaned_tasks_df)

# Prepare for training the attributions dataframe with fixed task names 
def prepare_df_for_training(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()

    # Drop rows with nulls in "TaskDescription"
    clean_df = clean_df.dropna(subset=["TaskDescription"])

    # Drop duplicates
    clean_df = clean_df.drop_duplicates()

    # Keep only "Task" and "TaskDescription" columns
    clean_df = clean_df[["FullName", "Task", "TaskDescription"]]
    
    return clean_df

clean_df = prepare_df_for_training(cleaned_tasks_df)

print(f"Number of dropped rows: {len(cleaned_tasks_df) - len(clean_df)}")
print(f"Number of remaining rows: {len(clean_df)}")

Number of dropped rows: 91329
Number of remaining rows: 40462


In [19]:
clean_df.head()

,FullName,Task,TaskDescription
8,Lauri Suominen,conceptualization,Was a part of brainstorming for the project id...
9,Lauri Suominen,visualization,Designed the wiki layout and look. Was a part ...
10,Lauri Suominen,software,Planned and coded the wiki.
11,Lauri Suominen,project administration,Organized and supervised IT subteam.
12,Lauri Suominen,fundraising,Took part in fundraising from the universities...


In [31]:
# Print number of rows for each task class 

for task in TASK_CLASSES:
    count = len(clean_df[clean_df["Task"] == task])
    print(f"Number of rows for task '{task}': {count}")

Number of rows for task 'writing': 6366
Number of rows for task 'entrepreneurship': 680
Number of rows for task 'visualization': 3175
Number of rows for task 'hardware': 416
Number of rows for task 'background research': 4253
Number of rows for task 'safety': 1713
Number of rows for task 'public engagement': 3770
Number of rows for task 'analysis': 3330
Number of rows for task 'software': 2082
Number of rows for task 'investigation': 4134
Number of rows for task 'fundraising': 1605
Number of rows for task 'data curation': 1187
Number of rows for task 'project administration': 2872
Number of rows for task 'lab maintenance': 0
Number of rows for task 'conceptualization': 4879


In [32]:
# Save file

clean_df.to_csv(OUTPUT_TSV, sep="\t", index=False)

In [33]:
# New dataframe where we merge tasks and task descriptions per FullName
# Merge Task into list of tasks per FullName, and TaskDescription into merged string
# So, one row per list of task names and TaskDescription in one row is related to multiple task names

merged_tasks_df = clean_df.groupby("FullName").agg({
    "Task": lambda x: list(x.unique()),
    "TaskDescription": lambda x: " ".join(x.unique())
}).reset_index()

merged_tasks_df = merged_tasks_df[["TaskDescription", "Task"]]
# Rename Task column to Tasks bc it is a list now
merged_tasks_df = merged_tasks_df.rename(columns={"Task": "Tasks"})
merged_tasks_df.to_csv(OUTPUT_MERGED_TASKS_TSV, sep="\t", index=False)